## Importing libraries and dataset

In [1]:
import pandas as pd
import numpy as np
import csv
import re
import torch
import torch.nn as nn

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

In [2]:


LABELS = [
    "political",
    "racial/ethnic",
    "religious",
    "gender/sexual",
    "other"
]

rows = []

with open(
    "../dataset/task2_polarization_type_no_id.txt",
    "r",
    encoding="utf-8",
    newline=""
) as f:
    reader = csv.reader(
        f,
        delimiter="|",
        quotechar='"'
    )

    header = next(reader)

    for row in reader:
        if len(row) != 6:
            continue

        text = row[0].strip()

        try:
            labels = [int(x) for x in row[1:]]
        except ValueError:
            continue

        if all(x in (0, 1) for x in labels):
            rows.append([text] + labels)

df = pd.DataFrame(rows, columns=["text"] + LABELS)

print("Dataset shape:", df.shape)
print(df[LABELS].sum())
print("Missing labels:", df[LABELS].isna().sum().sum())

Dataset shape: (9034, 6)
political        2915
racial/ethnic     862
religious         883
gender/sexual     697
other            2000
dtype: int64
Missing labels: 0


In [3]:
print("Rows before cleaning:", len(df))

df = df.dropna(subset=LABELS).copy()

print("Rows after cleaning:", len(df))
print("Remaining missing labels:", df[LABELS].isna().sum().sum())

Rows before cleaning: 9034
Rows after cleaning: 9034
Remaining missing labels: 0


In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

def tokenize(text):
    return text.split()

PAD_IDX = 0
UNK_IDX = 1

counter = Counter()

for text in train_df["text"]:
    counter.update(tokenize(text))

vocab = {
    "<PAD>": PAD_IDX,
    "<UNK>": UNK_IDX
}

for token, count in counter.items():
    if count >= 1:
        vocab[token] = len(vocab)

print("Vocabulary size:", len(vocab))
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Vocabulary size: 44609
Train: 7227
Validation: 903
Test: 904


In [5]:
class PolarDataset(Dataset):
    def __init__(self, dataframe, vocab):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe[LABELS].values
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenize(self.texts[idx])

        token_ids = [
            self.vocab.get(token, UNK_IDX)
            for token in tokens
        ]

        # Avoid empty sequences
        if not token_ids:
            token_ids = [UNK_IDX]

        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )


def collate_fn(batch):
    sequences, labels = zip(*batch)

    lengths = torch.tensor(
        [len(seq) for seq in sequences],
        dtype=torch.long
    )

    padded = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=PAD_IDX
    )

    labels = torch.stack(labels)

    return padded, lengths, labels


train_loader = DataLoader(
    PolarDataset(train_df, vocab),
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    PolarDataset(val_df, vocab),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    PolarDataset(test_df, vocab),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [6]:

for name, loader in [
    ("Train", train_loader),
    ("Validation", val_loader),
    ("Test", test_loader),
]:
    for batch_idx, (_, _, labels) in enumerate(loader):
        assert torch.isfinite(labels).all(), (
            f"{name}: invalid labels in batch {batch_idx}"
        )

    print(f"{name}: all labels are finite")


Train: all labels are finite
Validation: all labels are finite
Test: all labels are finite


In [7]:
assert df[LABELS].isin([0, 1]).all().all()
assert df[LABELS].notna().all().all()

print("Dataset labels are valid.")

Dataset labels are valid.


In [8]:
class BiLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=128,
        num_classes=5,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        # Last forward and backward hidden states
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        combined = torch.cat(
            (forward_hidden, backward_hidden),
            dim=1
        )

        combined = self.dropout(combined)

        # Return logits, not sigmoid probabilities
        return self.fc(combined)

In [9]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Create a completely new model
model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    num_classes=5,
    dropout=0.3
).to(device)

# Recreate the loss and optimizer
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

# Verify initialization
for name, param in model.named_parameters():
    assert torch.isfinite(param).all(), (
        f"Invalid parameter after initialization: {name}"
    )

print("All model parameters are finite.")

All model parameters are finite.


In [10]:
model.eval()

x, lengths, y = next(iter(train_loader))

x = x.to(device)
y = y.to(device)

with torch.no_grad():
    logits = model(x, lengths)
    loss = criterion(logits, y)

print("Logits finite:", torch.isfinite(logits).all().item())
print("Initial loss:", loss.item())

Logits finite: True
Initial loss: 0.6937462091445923


In [11]:
def run_epoch(loader, training=False):
    model.train(training)
    total_loss = 0.0

    for batch_idx, (x, lengths, y) in enumerate(loader):
        x = x.to(device)
        y = y.to(device)

        if not torch.isfinite(y).all():
            raise ValueError(
                f"Invalid labels in batch {batch_idx}"
            )

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(x, lengths)

            if not torch.isfinite(logits).all():
                raise ValueError(
                    f"NaN/Inf logits at batch {batch_idx}"
                )

            loss = criterion(logits, y)

            if not torch.isfinite(loss):
                raise ValueError(
                    f"NaN/Inf loss at batch {batch_idx}: "
                    f"{loss.item()}"
                )

            if training:
                loss.backward()

                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                    error_if_nonfinite=True
                )

                optimizer.step()

                # Detect invalid parameter updates
                if not all(
                    torch.isfinite(p).all()
                    for p in model.parameters()
                ):
                    raise ValueError(
                        "Model parameters became NaN/Inf"
                    )

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)


best_val_loss = float("inf")
best_state = None
patience = 5
wait = 0

for epoch in range(30):
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

        wait = 0
    else:
        wait += 1

    if wait >= patience:
        print("Early stopping")
        break

if best_state is None:
    raise RuntimeError("No valid model checkpoint was created.")

model.load_state_dict(best_state)

# torch.save(model.state_dict(), "bilstm_model.pth")
# print("Model saved successfully.")

Epoch 01 | Train loss: 0.5835 | Val loss: 0.4390
Epoch 02 | Train loss: 0.4385 | Val loss: 0.4292
Epoch 03 | Train loss: 0.4269 | Val loss: 0.4247
Epoch 04 | Train loss: 0.4189 | Val loss: 0.4217
Epoch 05 | Train loss: 0.4125 | Val loss: 0.4187
Epoch 06 | Train loss: 0.4051 | Val loss: 0.4165
Epoch 07 | Train loss: 0.3953 | Val loss: 0.4157
Epoch 08 | Train loss: 0.3838 | Val loss: 0.4148
Epoch 09 | Train loss: 0.3726 | Val loss: 0.4147
Epoch 10 | Train loss: 0.3595 | Val loss: 0.4159
Epoch 11 | Train loss: 0.3482 | Val loss: 0.4227
Epoch 12 | Train loss: 0.3362 | Val loss: 0.4306
Epoch 13 | Train loss: 0.3244 | Val loss: 0.4285
Epoch 14 | Train loss: 0.3123 | Val loss: 0.4345
Early stopping


<All keys matched successfully>

In [17]:
def get_probs(loader):
    model.eval()
    probs, targets = [], []
    with torch.no_grad():
        for x, lengths, y in loader:
            logits = model(x.to(device), lengths)
            probs.append(torch.sigmoid(logits).cpu().numpy())
            targets.append(y.numpy())
    return np.concatenate(probs), np.concatenate(targets)

val_probs, val_targets = get_probs(val_loader)
test_probs, test_targets = get_probs(test_loader)

# Find the best threshold per label on the VALIDATION set
best_thr = []
for i, label in enumerate(LABELS):
    scores = [
        (f1_score(val_targets[:, i], val_probs[:, i] >= t, zero_division=0), t)
        for t in np.arange(0.05, 0.60, 0.01)
    ]
    best_f1, thr = max(scores)
    best_thr.append(thr)
    print(f"{label:15s} threshold={thr:.2f}  val F1={best_f1:.3f}")

best_thr = np.array(best_thr)

# Apply those thresholds to the TEST set
test_preds = (test_probs >= best_thr).astype(int)

print(classification_report(
    test_targets, test_preds,
    target_names=LABELS, zero_division=0
))

political       threshold=0.11  val F1=0.527
racial/ethnic   threshold=0.12  val F1=0.194
religious       threshold=0.12  val F1=0.265
gender/sexual   threshold=0.13  val F1=0.175
other           threshold=0.28  val F1=0.397
               precision    recall  f1-score   support

    political       0.35      0.96      0.51       300
racial/ethnic       0.09      0.53      0.16        68
    religious       0.20      0.81      0.32       111
gender/sexual       0.10      0.41      0.16        68
        other       0.28      0.72      0.40       203

    micro avg       0.24      0.78      0.36       750
    macro avg       0.20      0.69      0.31       750
 weighted avg       0.26      0.78      0.39       750
  samples avg       0.29      0.57      0.35       750



In [25]:
print("Per-label classification report (multi-label setting):")

for i, label in enumerate(LABELS):
    print(f"\n--- {label} ---")
    print(classification_report(
        test_targets[:, i],
        test_preds[:, i],
        labels=[0, 1],
        target_names=["negative", "positive"],
        zero_division=0
    ))

print("\nOverall micro-averaged metrics:")
print({
    "precision": precision_score(test_targets, test_preds, average="micro", zero_division=0),
    "recall": recall_score(test_targets, test_preds, average="micro", zero_division=0),
    "f1": f1_score(test_targets, test_preds, average="micro", zero_division=0),
    "support": int(test_targets.sum()),
})


Per-label classification report (multi-label setting):

--- political ---
              precision    recall  f1-score   support

    negative       0.84      0.11      0.20       604
    positive       0.35      0.96      0.51       300

    accuracy                           0.39       904
   macro avg       0.60      0.54      0.36       904
weighted avg       0.68      0.39      0.30       904


--- racial/ethnic ---
              precision    recall  f1-score   support

    negative       0.94      0.57      0.71       836
    positive       0.09      0.53      0.16        68

    accuracy                           0.57       904
   macro avg       0.51      0.55      0.43       904
weighted avg       0.87      0.57      0.67       904


--- religious ---
              precision    recall  f1-score   support

    negative       0.95      0.55      0.70       793
    positive       0.20      0.81      0.32       111

    accuracy                           0.58       904
   macro avg

In [26]:

print("Label analysis")
print("-" * 75)

for i, label in enumerate(LABELS):
    print(
        f"{label:15s} | "
        f"Actual positives: {int(test_targets[:, i].sum()):4d} | "
        f"Predicted positives: {int(test_preds[:, i].sum()):4d} | "
        f"Mean probability: {test_probs[:, i].mean():.4f} | "
        f"Max probability: {test_probs[:, i].max():.4f}"
    )

Label analysis
---------------------------------------------------------------------------
political       | Actual positives:  300 | Predicted positives:  822 | Mean probability: 0.2294 | Max probability: 0.8535
racial/ethnic   | Actual positives:   68 | Predicted positives:  392 | Mean probability: 0.1272 | Max probability: 0.4143
religious       | Actual positives:  111 | Predicted positives:  446 | Mean probability: 0.1212 | Max probability: 0.3864
gender/sexual   | Actual positives:   68 | Predicted positives:  290 | Mean probability: 0.1195 | Max probability: 0.3876
other           | Actual positives:  203 | Predicted positives:  521 | Mean probability: 0.2892 | Max probability: 0.5607
